# Residual Norm Decomposition

Decompose the residual stream norm into per-component contributions:
norm buildup, direction decomposition, cross-position profiles, and interference.

## Why This Matters

Composition residual norm decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_norm_decomposition import (
    norm_contribution_by_component, layerwise_norm_buildup,
    norm_direction_decomposition, cross_position_norm_profile,
    component_interference_matrix,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Norm Contribution by Component

How much does each component contribute to ||residual||^2?

In [ ]:
result = norm_contribution_by_component(model, tokens, position=-1)
print(f"Total residual norm: {result['total_norm']:.4f}\n")
for c in result['per_component']:
    bar = '█' * int(c['fraction_of_total'] * 30)
    print(f"  {c['component']:8s}: norm={c['self_norm']:.4f}, "
          f"frac={c['fraction_of_total']:.1%}, "
          f"proj={c['projection_onto_residual']:+.4f} {bar}")

## Layerwise Norm Buildup

How does the residual norm grow step by step?

In [ ]:
result = layerwise_norm_buildup(model, tokens, position=-1)
for step in result['steps']:
    bar = '█' * int(step['norm'] / result['final_norm'] * 20)
    delta = f", Δ={step.get('delta_norm', 0):.4f}" if 'delta_norm' in step else ''
    print(f"  {step['step']:8s}: norm={step['norm']:.4f}{delta} {bar}")

## Direction Decomposition

Which components push the residual in its final direction?

In [ ]:
result = norm_direction_decomposition(model, tokens, position=-1)
print(f"Constructive components: {result['n_constructive']}/{len(result['per_component'])}\n")
for c in result['per_component']:
    sign = '+' if c['is_constructive'] else '-'
    print(f"  [{sign}] {c['component']:8s}: proj={c['projection']:+.4f}, "
          f"cos={c['cosine_with_residual']:+.4f}")

## Cross-Position Norm Profile

Do some positions have unusually large or small residual norms?

In [ ]:
result = cross_position_norm_profile(model, tokens)
print(f"Mean norm: {result['mean_norm']:.4f} ± {result['std_norm']:.4f}")
print(f"Outliers: {result['n_outliers']}\n")
for p in result['per_position']:
    outlier = ' [OUTLIER]' if p['is_outlier'] else ''
    bar = '█' * int(p['norm'] / result['mean_norm'] * 10)
    print(f"  Pos {p['position']} (tok {p['token']:2d}): "
          f"norm={p['norm']:.4f}, z={p['z_score']:+.2f}{outlier} {bar}")

## Component Interference Matrix

Which components interfere constructively or destructively?

In [ ]:
result = component_interference_matrix(model, tokens, position=-1)
print(f"Total constructive: {result['total_constructive']:.4f}")
print(f"Total destructive: {result['total_destructive']:.4f}\n")
for p in sorted(result['pairs'], key=lambda x: abs(x['dot_product']), reverse=True)[:8]:
    sign = '+' if p['dot_product'] > 0 else '-'
    print(f"  [{sign}] {p['component_a']:8s} ↔ {p['component_b']:8s}: "
          f"dot={p['dot_product']:+.4f}, cos={p['cosine']:+.4f}")